## Poking at OpenRouter's TTS Models

Companion notebook to https://github.com/kasir-barati/smart-novel-beatrice/issues/6. `Qwen3TtsProvider.get_voices()` does `GET {base_url}{voices_path}` (default `/v1/voices`, DeepInfra's route), and that 404 against OpenRouter is happening in `generateAudio` method, to be precise `resolve_audio_voices()` fails before publishing to RabbitMQ. Here I wanna find out, hands-on, whether OpenRouter really has *no* way to list voices for a model, whether it's just shaped differently than DeepInfra's endpoint, what about Gemini-TTS? I have the feeling I need a OpenRouter provider instead and the current `Qwen3TtsProvider` must be refactored to be exactly only compatible with the Alibaba's API which is not OpenAI-Compatible unfortunately. Same story for `GeminiTtsProvider`. 

BTW for this notebook you need `OPENROUTER_API_KEY` in `.env`.

In [19]:
from os import getenv
from typing import cast

import httpx
from dotenv import load_dotenv


load_dotenv(override=True)

openrouter_api_key = cast(str, getenv("OPENROUTER_API_KEY"))
if not openrouter_api_key:
    raise ValueError("OPENROUTER_API_KEY is not set in the environment variables.")

OPENROUTER_URL = "https://openrouter.ai/api"
HEADERS = {"Authorization": f"Bearer {openrouter_api_key}"}

QWEN3_TTS_MODEL = "qwen/qwen-audio-3.0-tts-flash"
GEMINI_TTS_MODEL = "google/gemini-3.1-flash-tts-preview"

### Reproducing The 404

Same request we send in `Qwen3TtsProvider.get_voices()` and `GeminiTtsProvider.get_voices()`. This is the one that breaks `generateAudio` end-to-end when pointed at OpenRouter.

In [2]:
async with httpx.AsyncClient(base_url=OPENROUTER_URL, headers=HEADERS) as client:
    for path in ("/v1/voices", "/v1/audio/voices"):
        response = await client.get(path)
        print(f"GET {path} -> {response.status_code}")

GET /v1/voices -> 404
GET /v1/audio/voices -> 404


### Is There *any* Voice-listing Shape at All?

[OpenRouter's docs for Qwen3-TTS](https://openrouter.ai/qwen/qwen-audio-3.0-tts-flash-20260723) don't document a `GET /v1/voices` route the way [DeepInfra](https://deepinfra.com/Qwen/Qwen3-TTS/api?example=list-voices-http) does. Instead, model discovery goes through [`GET /v1/models`](https://openrouter.ai/docs/api/api-reference/models/list-all-models-and-their-properties), and the `Model` schema in their OpenAPI spec carries a `supported_voices: array | null` field. So the voices aren't listed at a dedicated endpoint!

They're metadata on the model object itself. Let's check if that field is actually populated for our model, since `array | null` means it might just be `null` in
practice.

In [20]:
async with httpx.AsyncClient(base_url=OPENROUTER_URL, headers=HEADERS) as client:
    response = await client.get("/v1/models", params={"output_modalities": "speech"})
    response.raise_for_status()
    models = response.json()["data"]

print(f"{len(models)} TTS model(s) returned")

matched_qwen3_tts_model_spec = cast(dict, next((m for m in models if m["id"] == QWEN3_TTS_MODEL), None))
matched_gemini_tts_model_spec = cast(dict, next((m for m in models if m["id"] == GEMINI_TTS_MODEL), None))

if matched_qwen3_tts_model_spec is None or matched_gemini_tts_model_spec is None:
    raise ValueError(f"One or both models not found in output_modalities=speech listing")

# Model: qwen/qwen-audio-3.0-tts-flash, supported_voices: ['loongjohn', 'longanhuan_v3.6']
print(f"Model: {matched_qwen3_tts_model_spec['id']}, supported_voices: {matched_qwen3_tts_model_spec.get('supported_voices')}")

# Model: google/gemini-3.1-flash-tts-preview, supported_voices: ['Zephyr', 'Puck', 'Charon', 'Kore', 'Fenrir', 'Leda', 'Orus', 'Aoede', 'Callirrhoe', 'Autonoe', 'Enceladus', 'Iapetus', 'Umbriel', 'Algieba', 'Despina', 'Erinome', 'Algenib', 'Rasalgethi', 'Laomedeia', 'Achernar', 'Alnilam', 'Schedar', 'Gacrux', 'Pulcherrima', 'Achird', 'Zubenelgenubi', 'Vindemiatrix', 'Sadachbia', 'Sadaltager', 'Sulafat']
print(f"Model: {matched_gemini_tts_model_spec['id']}, supported_voices: {matched_gemini_tts_model_spec.get('supported_voices')}")

18 TTS model(s) returned
Model: qwen/qwen-audio-3.0-tts-flash, supported_voices: ['loongjohn', 'longanhuan_v3.6']
Model: google/gemini-3.1-flash-tts-preview, supported_voices: ['Zephyr', 'Puck', 'Charon', 'Kore', 'Fenrir', 'Leda', 'Orus', 'Aoede', 'Callirrhoe', 'Autonoe', 'Enceladus', 'Iapetus', 'Umbriel', 'Algieba', 'Despina', 'Erinome', 'Algenib', 'Rasalgethi', 'Laomedeia', 'Achernar', 'Alnilam', 'Schedar', 'Gacrux', 'Pulcherrima', 'Achird', 'Zubenelgenubi', 'Vindemiatrix', 'Sadachbia', 'Sadaltager', 'Sulafat']


### The Per-model endpoints route

There's also [`GET /v1/models/{author}/{model-name}/endpoints`](https://openrouter.ai/docs/api/api-reference/endpoints/list-all-endpoints-for-a-model) you can find in the response you got from the previous step, instead of `supported_voices` try to get `links`. This is meant for inspecting which upstream providers serve a model (used for STT model discovery in OpenRouter's own docs). Checking whether `supported_voices` shows up there too, or only on the bulk `/v1/models`
listing above.

In [ ]:
QWEN3_TTS_PER_MODEL_ENDPOINT = cast(str, matched_qwen3_tts_model_spec.get("links", {}).get("details")).removeprefix('/api')
GEMINI_TTS_PER_MODEL_ENDPOINT = cast(str, matched_gemini_tts_model_spec.get("links", {}).get("details")).removeprefix('/api')


async with httpx.AsyncClient(base_url=OPENROUTER_URL, headers=HEADERS) as client:
    response = await client.get(QWEN3_TTS_PER_MODEL_ENDPOINT)
    print(f"GET {QWEN3_TTS_PER_MODEL_ENDPOINT}")
    print(response.json())

    response = await client.get(GEMINI_TTS_PER_MODEL_ENDPOINT)
    print(f"GET {GEMINI_TTS_PER_MODEL_ENDPOINT} ")
    print(response.json())

GET /v1/models/qwen/qwen-audio-3.0-tts-flash-20260723/endpoints
{'data': {'id': 'qwen/qwen-audio-3.0-tts-flash', 'name': 'Qwen: Qwen-Audio-3.0-TTS Flash', 'created': 1784817207, 'description': "Qwen-Audio-3.0-TTS Flash is Alibaba's fast, cost-efficient text-to-speech model, generating spoken audio from text via the DashScope Speech Synthesizer API.", 'architecture': {'tokenizer': 'Other', 'instruct_type': None, 'modality': 'text->speech', 'input_modalities': ['text'], 'output_modalities': ['speech']}, 'endpoints': [{'name': 'Alibaba | qwen/qwen-audio-3.0-tts-flash-20260723', 'model_id': 'qwen/qwen-audio-3.0-tts-flash', 'model_name': 'Qwen: Qwen-Audio-3.0-TTS Flash', 'context_length': 0, 'pricing': {'prompt': '0.000015', 'completion': '0', 'discount': 0}, 'provider_name': 'Alibaba', 'tag': 'alibaba', 'quantization': 'unknown', 'max_completion_tokens': 0, 'max_prompt_tokens': None, 'supported_parameters': ['max_tokens', 'temperature', 'top_p', 'seed', 'presence_penalty', 'response_format

#### Should Print

```js
{
  data: {
    id: "qwen/qwen-audio-3.0-tts-flash",
    name: "Qwen: Qwen-Audio-3.0-TTS Flash",
    created: 1784817207,
    description:
      "Qwen-Audio-3.0-TTS Flash is Alibaba's fast, cost-efficient text-to-speech model, generating spoken audio from text via the DashScope Speech Synthesizer API.",
    architecture: {
      tokenizer: "Other",
      instruct_type: null,
      modality: "text->speech",
      input_modalities: ["text"],
      output_modalities: ["speech"],
    },
    endpoints: [
      {
        name: "Alibaba | qwen/qwen-audio-3.0-tts-flash-20260723",
        model_id: "qwen/qwen-audio-3.0-tts-flash",
        model_name: "Qwen: Qwen-Audio-3.0-TTS Flash",
        context_length: 0,
        pricing: { prompt: "0.000015", completion: "0", discount: 0 },
        provider_name: "Alibaba",
        tag: "alibaba",
        quantization: "unknown",
        max_completion_tokens: 0,
        max_prompt_tokens: null,
        supported_parameters: [
          "max_tokens",
          "temperature",
          "top_p",
          "seed",
          "presence_penalty",
          "response_format",
        ],
        supports_tool_choice: {
          none: true,
          auto: true,
          required: true,
          function: true,
        },
        status: 0,
        uptime_last_30m: null,
        uptime_last_5m: null,
        uptime_last_1d: 100,
        supports_implicit_caching: false,
        supports_voice_cloning: false,
        latency_last_30m: null,
        throughput_last_30m: null,
      },
    ],
  },
}
```

And Gemini:

```js
{
  data: {
    id: "google/gemini-3.1-flash-tts-preview",
    name: "Google: Gemini 3.1 Flash TTS Preview",
    created: 1776999308,
    description:
      "Gemini 3.1 Flash TTS Preview is a text-to-speech model from Google, and a substantial generational step up from Gemini 2.5 Flash TTS. It takes text input and produces audio output...",
    architecture: {
      tokenizer: "Gemini",
      instruct_type: null,
      modality: "text->speech",
      input_modalities: ["text"],
      output_modalities: ["speech"],
    },
    endpoints: [
      {
        name: "Google | google/gemini-3.1-flash-tts-preview",
        model_id: "google/gemini-3.1-flash-tts-preview",
        model_name: "Google: Gemini 3.1 Flash TTS Preview",
        context_length: 32768,
        pricing: { prompt: "0.000001", completion: "0.00002", discount: 0 },
        provider_name: "Google",
        tag: "google-vertex",
        quantization: "unknown",
        max_completion_tokens: 16384,
        max_prompt_tokens: 8192,
        supported_parameters: [
          "max_tokens",
          "temperature",
          "top_p",
          "seed",
          "response_format",
        ],
        supports_tool_choice: {
          none: true,
          auto: true,
          required: true,
          function: true,
        },
        status: 0,
        uptime_last_30m: 100,
        uptime_last_5m: 100,
        uptime_last_1d: 100,
        supports_implicit_caching: false,
        supports_voice_cloning: false,
        latency_last_30m: null,
        throughput_last_30m: null,
      },
    ],
  },
};
```

### Synthesize With a Discovered Voice

Using whatever `supported_voices` returned above (falls back to `loongjohn`, confirmed working in the bug report, if the field came back empty).

In [25]:
import time
from pathlib import Path

import miniaudio

out = Path("tts_output.mp3")
supported_voices = matched_qwen3_tts_model_spec.get("supported_voices") or ["loongjohn"]
voice = supported_voices[0]

print(f"using voice: {voice}")

async with httpx.AsyncClient(base_url=OPENROUTER_URL, headers=HEADERS, timeout=60) as client:
    response = await client.post(
        "/v1/audio/speech",
        json={
            "model": QWEN3_TTS_MODEL,
            "input": "Testing OpenRouter's Qwen TTS voice list.",
            "voice": voice,
            "response_format": "mp3",
        },
    )
    response.raise_for_status()

audio_bytes = response.content
out.write_bytes(audio_bytes)

print(f"content-type: {response.headers.get('Content-Type')}")
print(f"{len(audio_bytes)} bytes, generation id: {response.headers.get('X-Generation-Id')}")

# IPython.display.Audio's <audio> tag renders fine but VS Code's notebook webview
# doesn't reliably play it, so play straight through the OS device instead.
info = miniaudio.get_file_info(str(out))
with miniaudio.PlaybackDevice(
    output_format=miniaudio.SampleFormat.SIGNED16,
    nchannels=info.nchannels,
    sample_rate=info.sample_rate,
) as device:
    stream = miniaudio.stream_file(
        str(out),
        output_format=miniaudio.SampleFormat.SIGNED16,
        nchannels=info.nchannels,
        sample_rate=info.sample_rate,
    )
    next(stream)
    device.start(stream)
    time.sleep(info.duration + 0.3)

using voice: loongjohn
content-type: audio/mpeg
94171 bytes, generation id: gen-tts-1789035863-N7OAEiGvCkGgDOa4RT4D


### What Happens With a Voice that isn't in `supported_voices`?

If OpenRouter rejects an invalid voice at synthesis time with a clear error, that's a second safety net even without pre-validating client-side, worth knowing regardless of which fix we pick.

In [ ]:
async with httpx.AsyncClient(base_url=OPENROUTER_URL, headers=HEADERS, timeout=60) as client:
    response = await client.post(
        "/v1/audio/speech",
        json={
            "model": QWEN3_TTS_MODEL,
            "input": "This should fail.",
            "voice": "definitely-not-a-real-voice",
            "response_format": "mp3",
        },
    )
    print(f"POST /v1/audio/speech (bad voice) -> {response.status_code}")
    print(response.text)

## Findings

- **Confirmed the bug report:** both `GET /v1/voices` and `GET /v1/audio/voices` 404 against OpenRouter. There is no DeepInfra-style dedicated voices-listing endpoint.
- **But OpenRouter does expose voices.** Their `GET /v1/models` response includes a `supported_voices: array | null` field per model. Filtering that listing for `qwen/qwen-audio-3.0-tts-flash` and reading `supported_voices` off the matching entry is the actual OpenRouter equivalent of DeepInfra's `GET /v1/voices`. It's just model metadata, not a standalone REST resource.
- `supported_voices` actually came back populated for both Gemini-TTS and Qwen3-TTS models.
- Calling the `POST /v1/audio/speech` with an invalid voice name will fail with a 400 status code!
- **Not passing `response_format` in the synthesis request returns raw PCM, not MP3.** The file saved as `tts_output.mp3` without it wasn't playable because `IPython.display.Audio` trusts the `.mp3` filename extension (`mimetypes.guess_type`) rather than sniffing the bytes, so it labelled raw PCM as `audio/mpeg`. Passing `"response_format": "mp3"` explicitly fixes it.

**Remove `Qwen3TtsProvider` and the current `GeminiTtsProvider`.** Instead of what we are doing right now we must introduce a new `OpenRouterProvider` which implement it like it should. So when we call `get_voices` it calls get a model by its slug. I guess we can do some magic like this there:

```py
model_author, model_slug = QWEN3_TTS_MODEL.split('/', 1)
# Same thing for Gemeni works
```

This way we at least know we are only have OpenRouter provider.

> [!CAUTION]
>
> You must also update the mocked wrapper around the Qwen3-TTS we have in our local machine. and call the thin wrapper an OpenRouter compatible wrapper.